# API wrappers

The OpenWeatherMap offers REST endpoints for querying current weather, forecasts, historical data, etc. However, accessing this data directly via the REST API requires handling multiple API calls, query parameters, and response parsing. The pyowm library abstracts these complexities and provides useful built-in functionalities.

After signing in to OpenWeatherMap retrieve your api key at https://home.openweathermap.org/api_keys

You will also need to install the pyowm package: pip install pyowm 

In [11]:
import requests
import pyowm
import json
from getpass import getpass



## use case 1: managing API keys

In a raw rest API call you always have to manage credentials in each individual call. Wrappers usually store and manage the authentication for you

In [22]:
api_key = getpass("Enter your OpenWeatherMap API key: ")

#You can get current weather data by making a GET request to an endpoint like:

params = {
    'appid' : api_key
}

response = requests.get('https://api.openweathermap.org/data/2.5/weather?q=London', params = params)

json.loads(response.text)

#but for every call you make using GET from now on you do need to add the parameters, since the raw API does not manage authentication for you

{'coord': {'lon': -0.1257, 'lat': 51.5085},
 'weather': [{'id': 804,
   'main': 'Clouds',
   'description': 'overcast clouds',
   'icon': '04d'}],
 'base': 'stations',
 'main': {'temp': 284.89,
  'feels_like': 284.29,
  'temp_min': 284.1,
  'temp_max': 285.21,
  'pressure': 1018,
  'humidity': 83,
  'sea_level': 1018,
  'grnd_level': 1014},
 'visibility': 10000,
 'wind': {'speed': 5.14, 'deg': 220},
 'clouds': {'all': 100},
 'dt': 1774619942,
 'sys': {'type': 2,
  'id': 2075535,
  'country': 'GB',
  'sunrise': 1774590452,
  'sunset': 1774635846},
 'timezone': 0,
 'id': 2643743,
 'name': 'London',
 'cod': 200}

Most wrappers (pyowm included) include some way of initializing a session with the authentication key that you then don't need to type again.

Initialize pyowm with the default configuration. Thenopen the weather manager

Check out a snippet here: https://pyowm.readthedocs.io/en/latest/v3/code-recipes.html#weather_data

## use case 2: Simplified calls

With the raw REST API, you'd have to build a URL manually, send the request, and parse the JSON response to get the current weather.

In [29]:
city = 'kerkrade'
url = f'http://api.openweathermap.org/data/2.5/weather?q={city}'

response = requests.get(url,params= params)
data = response.json()
temperature = data['main']['temp']
humidity = data['main']['humidity']
wind_speed = data['wind']['speed']

print(f"Temperature: {temperature}°C, Humidity: {humidity}%, Wind Speed: {wind_speed} m/s")

Temperature: 281.48°C, Humidity: 53%, Wind Speed: 2.57 m/s


Get the equivalent call as above for the city of London using the pyowm package

In [27]:
from pyowm import OWM

owm = OWM(api_key)
weather_manager = owm.weather_manager()

weather = weather_manager.weather_at_place("vaals").weather

print(weather.temperature('celsius'))

{'temp': 7.99, 'temp_max': 8.65, 'temp_min': 7.2, 'feels_like': 6.65, 'temp_kf': None}


## use case 3: Combining and chaining calls

Wrappers often offer methods that make multiple calls to batch requests that make sense to batch. And often they offer methods that make sequences of calls that each returns information necessary to make the next call.

For example, to get a weather forecast for a specific city using the raw API you need to first geocode the city to get its latitude and longitude:

In [32]:
city = 'Vaals'
geocode_url = f'http://api.openweathermap.org/data/2.5/weather?q={city}'
geocode_response = requests.get(geocode_url,params=params).json()

lat = geocode_response['coord']['lat']
lon = geocode_response['coord']['lon']

Then, request the weather forecast for that latitude/longitude:

In [33]:
forecast_url = f'http://api.openweathermap.org/data/2.5/forecast?lat={lat}&lon={lon}'
forecast_response = requests.get(forecast_url, params=params).json()

for entry in forecast_response['list']:
    print(f"Time: {entry['dt_txt']}, Temp: {entry['main']['temp']}°C")

Time: 2026-03-27 15:00:00, Temp: 281.14°C
Time: 2026-03-27 18:00:00, Temp: 280.5°C
Time: 2026-03-27 21:00:00, Temp: 278.94°C
Time: 2026-03-28 00:00:00, Temp: 276.83°C
Time: 2026-03-28 03:00:00, Temp: 276.89°C
Time: 2026-03-28 06:00:00, Temp: 277.64°C
Time: 2026-03-28 09:00:00, Temp: 278.32°C
Time: 2026-03-28 12:00:00, Temp: 279.8°C
Time: 2026-03-28 15:00:00, Temp: 280.3°C
Time: 2026-03-28 18:00:00, Temp: 276.46°C
Time: 2026-03-28 21:00:00, Temp: 274.07°C
Time: 2026-03-29 00:00:00, Temp: 273.08°C
Time: 2026-03-29 03:00:00, Temp: 272.5°C
Time: 2026-03-29 06:00:00, Temp: 272.76°C
Time: 2026-03-29 09:00:00, Temp: 278.4°C
Time: 2026-03-29 12:00:00, Temp: 281.3°C
Time: 2026-03-29 15:00:00, Temp: 281.31°C
Time: 2026-03-29 18:00:00, Temp: 278.43°C
Time: 2026-03-29 21:00:00, Temp: 277.88°C
Time: 2026-03-30 00:00:00, Temp: 278.2°C
Time: 2026-03-30 03:00:00, Temp: 278.82°C
Time: 2026-03-30 06:00:00, Temp: 277.59°C
Time: 2026-03-30 09:00:00, Temp: 278.61°C
Time: 2026-03-30 12:00:00, Temp: 279.75°C

Two calls: one for geocoding, one for forecasts.
But with pyowm, because this is a common operation, there is a method that handles the geocoding internally and then fetches the weather forecast in one step.

Get the above forecast in a single call using pyowm.

Hint: search for "forecast_at_place" in the code recipies of the documentation

In [34]:
# Step 1: Get coordinates from city
city = "Vaals"

geo_url = "https://api.openweathermap.org/data/2.5/weather"

params = {
    "q": city,
    "appid": api_key
}

geo_response = requests.get(geo_url, params=params).json()

lat = geo_response["coord"]["lat"]
lon = geo_response["coord"]["lon"]

print(f"Coordinates: {lat}, {lon}")

Coordinates: 50.7708, 6.0181


In [35]:
# Step 2: Get forecast using lat/lon
forecast_url = "https://api.openweathermap.org/data/2.5/forecast"

params = {
    "lat": lat,
    "lon": lon,
    "appid": api_key,
    "units": "metric"
}

forecast_response = requests.get(forecast_url, params=params).json()

# Print forecast
for entry in forecast_response["list"][:5]:
    print(f"Time: {entry['dt_txt']}, Temp: {entry['main']['temp']}°C")

Time: 2026-03-27 15:00:00, Temp: 7.84°C
Time: 2026-03-27 18:00:00, Temp: 7.25°C
Time: 2026-03-27 21:00:00, Temp: 5.74°C
Time: 2026-03-28 00:00:00, Temp: 3.68°C
Time: 2026-03-28 03:00:00, Temp: 3.74°C


## use case 4: Convenience methods

Wrappers often offer built-in methods to handle common kinds of tasks related to the APIs, reducing the need for manual calculations.

for example converting units (e.g., temperature from Celsius to Fahrenheit) or working with more complex data requires manual conversion when using the raw API.

In [ ]:
city = 'London'
url = f'http://api.openweathermap.org/data/2.5/weather?q={city}&appid={api_key}'

response = requests.get(url)
data = response.json()
temperature_celsius = data['main']['temp']
temperature_fahrenheit = (temperature_celsius * 9/5) + 32

print(f"Temperature in Celsius: {temperature_celsius}°C, Fahrenheit: {temperature_fahrenheit}°F")

But the pyowm wrapper offers built-in methods to handle these kinds of tasks, reducing the need for manual calculations.
Get the temperature both in Celcius and Farenheit using pyowm. Navigate the code recipes to figure out the inbuilt methods for this.

In [36]:
owm = OWM(api_key)
mgr = owm.weather_manager()

weather = mgr.weather_at_place("Vaals").weather

# Get temperatures directly in different units
temp_celsius = weather.temperature('celsius')['temp']
temp_fahrenheit = weather.temperature('fahrenheit')['temp']

print(f"Temperature in Celsius: {temp_celsius}°C")
print(f"Temperature in Fahrenheit: {temp_fahrenheit}°F")

Temperature in Celsius: 7.71°C
Temperature in Fahrenheit: 45.88°F
